In [86]:
#pip install dominate

In [87]:
#from google.colab import drive
#drive.mount('/content/drive')

In [88]:
#from google.colab import files
#files.upload()

In [89]:
#!cp -r "/content/drive/MyDrive/Colab Notebooks/Paired MRI (T1, T2) and CT Scans Dataset" /content/

In [90]:
#!ls /content/

In [91]:
import sys
#del sys.modules['mymodel']
import importlib
import Metrics
import Helper_Functions
import Yang_2025  as mymodel

_=importlib.reload(Metrics)
_=importlib.reload(Helper_Functions)
_=importlib.reload(mymodel)

In [92]:
import torch
from torch import nn
from torchmetrics.image import StructuralSimilarityIndexMeasure

_=torch.manual_seed(111)  # Random Generator Seed

In [93]:
device=Helper_Functions.Device()
transform=Helper_Functions.Transform()

Device: cpu
False
No GPU


In [94]:
#Hyper Parameters
percent=0.8
batch_size = 2
lr = 0.001
num_epochs = 1
lambda_L1 = 100  
lambda_ssim = 10
beta1 = 0.5
beta2 = 0.999
root_dir=r"C:\Users\fhabi\Desktop\PhD\DataSet\New folder"

In [95]:
dataset = Helper_Functions.load_nii_files_fromDataset(root_dir)
train_dataset,test_dataset=Helper_Functions.Train_Test_Split_Dataset(dataset,percent)
train_loader,test_loader=Helper_Functions.Train_Test_DataLoader(train_dataset, test_dataset,batch_size)

Total samples: 3
Train samples: 2
Test  samples: 1


In [96]:
#Helper_Functions.Plot_Sample_of_Dataset(train_dataset)

In [97]:
discriminator = mymodel.Discriminator().to(device=device)

In [98]:
generator = mymodel.Generator().to(device=device)

In [99]:
loss_function = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()                      # L1 Loss
ssim_metric = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

def ssim_loss(fake, real):
    ssim_value = ssim_metric(fake, real)
    return 1 - ssim_value

In [100]:
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, beta2))

scheduler_G = torch.optim.lr_scheduler.LambdaLR(optimizer_generator, lr_lambda=Helper_Functions.lambda_lr)
scheduler_D = torch.optim.lr_scheduler.LambdaLR(optimizer_discriminator, lr_lambda=Helper_Functions.lambda_lr)

In [101]:
best_g_loss = float("inf")
best_d_loss = float("inf")

losses_d = []
losses_g = []
for epoch in range(num_epochs):
    for n, (MRI,real_PET) in enumerate(train_loader):
        real_PET = real_PET.to(device)
        MRI = MRI.to(device)
        # ---------------------
        # Train Discriminator
        # ---------------------

        discriminator.zero_grad()

        fake_PET = generator(MRI)
        # real pair  
        d_real = discriminator(MRI, real_PET)

        real_labels = torch.ones_like(d_real).to(device)

        loss_d_real = loss_function(d_real, real_labels)

        # fake pair
        d_fake = discriminator(MRI, fake_PET.detach())

        fake_labels = torch.zeros_like(d_fake).to(device)

        loss_d_fake = loss_function(d_fake, fake_labels)

        loss_discriminator = (loss_d_real + loss_d_fake) * 0.5

        loss_discriminator.backward()
        
        optimizer_discriminator.step()

        # ---------------------
        # Train Generator
        # ---------------------

        generator.zero_grad()

        fake_PET = generator(MRI)

        output = discriminator(MRI, fake_PET)

        real_labels = torch.ones_like(output).to(device)

        loss_g_gan = loss_function(output, real_labels)

        loss_g_l1 = l1_loss(fake_PET, real_PET) * lambda_L1

        loss_ssim_l = ssim_loss(fake_PET, real_PET) * lambda_ssim 

        loss_generator = loss_g_gan + loss_g_l1 +loss_ssim_l

        loss_generator.backward()

        optimizer_generator.step()

        # 👇 BEST MODEL SAVE HERE
        if loss_generator.item() < best_g_loss:
            best_g_loss = loss_generator.item()
            torch.save(generator.state_dict(), f"{mymodel.__name__}_best_generator.pth")

        if loss_discriminator.item() < best_d_loss:
            best_d_loss = loss_discriminator.item()
            torch.save(discriminator.state_dict(), f"{mymodel.__name__}_best_discriminator.pth")

        print(f"Pair Image: {n+1}")

    # update learning rate end of each epoch
    scheduler_G.step()
    scheduler_D.step()

    losses_d.append(loss_discriminator.item())
    losses_g.append(loss_generator.item())
    print(f"Epoch: {epoch+1}")

    if (epoch + 1) % 2 == 0:  # Each Two Epoch
      print(f"Epoch: {epoch+1}")
      print(f"LR D:   {scheduler_D.get_last_lr()[0]:.6f}")
      print(f"LR G:   {scheduler_G.get_last_lr()[0]:.6f}")
      print(f"Loss D: {loss_discriminator}")
      print(f"Loss G: {loss_generator}")
      
      Helper_Functions.Show(generator,epoch+1,train_dataset,device)
      Helper_Functions.Show_Loss(losses_g,losses_d)

Pair Image: 1
Epoch: 1


In [102]:
metrics = Metrics.evaluate_test_set(generator, test_loader, device)

MSE: 0.0285
MAE: 0.1246
PSNR: 21.4756
